# Entrenamiento, Evaluación y Selección del Modelo FAQ - UPQ

Este notebook implementa el flujo completo de aprendizaje supervisado para el **Chatbot de Preguntas Frecuentes (FAQ) de la Universidad Politécnica de Querétaro (UPQ)**:

1. **Carga y exploración del corpus** (`data/corpus_upq.csv`) con las columnas `texto`, `intencion` y `respuesta_oficial`.
2. **Preprocesamiento y normalización de texto** utilizando `limpiar_texto` del módulo `src/preprocess.py`.
3. **Partición estratificada 80/20** (`train_test_split` con `stratify=y` y `random_state=42`).
4. **Extracción de características con TF-IDF** ajustada **únicamente sobre el conjunto de entrenamiento** (`fit_transform` en train, `transform` en test) para evitar fuga de datos (*data leakage*).
5. **Entrenamiento y evaluación comparativa** de tres clasificadores supervisados:
   - `LogisticRegression`
   - `CalibratedClassifierCV(LinearSVC)`
   - `MultinomialNB`
6. **Visualización de matrices de confusión y tabla comparativa de F1-score macro**.
7. **Exportación por separado** del vectorizador (`models/vectorizador.pkl`) y del modelo ganador (`models/modelo.pkl`) mediante `joblib.dump`.

## 1. Importación de librerías y configuración de rutas

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Configurar la raíz del proyecto para importar src.preprocess
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.preprocess import limpiar_texto

DATA_PATH = BASE_DIR / "data" / "corpus_upq.csv"
MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"
VECTORIZER_OUTPUT_PATH = MODELS_DIR / "vectorizador.pkl"
MODEL_OUTPUT_PATH = MODELS_DIR / "modelo.pkl"
METRICS_OUTPUT_PATH = REPORTS_DIR / "metrics.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio base del proyecto: {BASE_DIR}")

## 2. Carga y exploración del corpus (`data/corpus_upq.csv`)

Verificamos que el dataset cumpla con el contrato de datos (`texto,intencion,respuesta_oficial`) y revisamos la distribución de ejemplos por intención.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el corpus en: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, encoding="utf-8")
columnas_requeridas = {"texto", "intencion", "respuesta_oficial"}
if not columnas_requeridas.issubset(df.columns):
    raise ValueError(f"El CSV debe contener las columnas: {columnas_requeridas}")

df = df.dropna(subset=["texto", "intencion", "respuesta_oficial"]).copy()
print(f"Total de registros cargados: {len(df)}")
print("\nDistribución por intención:")
print(df["intencion"].value_counts())
df.head()

## 3. Limpieza y preprocesamiento de texto

Aplicamos la función `limpiar_texto` de `src/preprocess.py` para convertir a minúsculas, eliminar acentos/diacríticos, remover signos de puntuación y normalizar espacios.

In [ ]:
df["texto_limpio"] = df["texto"].astype(str).apply(limpiar_texto)
X = df["texto_limpio"]
y = df["intencion"].astype(str)

df[["texto", "texto_limpio", "intencion"]].head(10)

## 4. División Estratificada 80/20 y Vectorización TF-IDF

Dividimos el corpus en **80% entrenamiento** y **20% prueba** conservando la proporción de cada clase (`stratify=y`).

Posteriormente ajustamos el `TfidfVectorizer(ngram_range=(1, 2))` **exclusivamente sobre `X_train`** y transformamos `X_test` con el vocabulario aprendido.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

vectorizador = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizador.fit_transform(X_train)
X_test_tfidf = vectorizador.transform(X_test)

print(f"Tamaño de entrenamiento: {X_train_tfidf.shape[0]} muestras | Vocabulario TF-IDF: {X_train_tfidf.shape[1]} términos")
print(f"Tamaño de prueba:        {X_test_tfidf.shape[0]} muestras")

## 5. Entrenamiento, Evaluación Comparativa y Matrices de Confusión

Entrenamos tres clasificadores con soporte para probabilidades (`predict_proba`):
- **LogisticRegression**: clasificador lineal regularizado con salida probabilística directa.
- **LinearSVC calibrado (`CalibratedClassifierCV`)**: máquina de vectores de soporte lineal con calibración de probabilidades mediante validación cruzada.
- **MultinomialNB**: clasificador Naive Bayes multinomial clásico para conteos y pesos TF-IDF.

Calculamos `Accuracy`, `Precision macro`, `Recall macro`, `F1-score macro` y graficamos la matriz de confusión de cada modelo.

In [ ]:
etiquetas = sorted(y.unique().tolist())

clasificadores = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "linear_svc_calibrated": CalibratedClassifierCV(LinearSVC(random_state=42), cv=3),
    "multinomial_nb": MultinomialNB(),
}

resultados = {}
modelos_entrenados = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nombre, clf) in zip(axes, clasificadores.items()):
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)

    accuracy = float(accuracy_score(y_test, y_pred))
    precision_macro = float(precision_score(y_test, y_pred, average="macro", zero_division=0))
    recall_macro = float(recall_score(y_test, y_pred, average="macro", zero_division=0))
    f1_macro = float(f1_score(y_test, y_pred, average="macro", zero_division=0))

    print("=" * 70)
    print(f"Modelo: {nombre}")
    print("-" * 70)
    print(classification_report(y_test, y_pred, zero_division=0))
    print(f"F1-score macro: {f1_macro:.4f}\n")

    cm = confusion_matrix(y_test, y_pred, labels=etiquetas)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=etiquetas,
        yticklabels=etiquetas,
        ax=ax,
    )
    ax.set_title(f"{nombre}\n(F1 macro = {f1_macro:.4f})")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

    # Guardar también cada matriz de confusión individual en reports/
    ruta_cm = REPORTS_DIR / f"confusion_matrix_{nombre}.png"
    fig_ind, ax_ind = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=etiquetas,
        yticklabels=etiquetas,
        ax=ax_ind,
    )
    ax_ind.set_title(f"Matriz de Confusión - {nombre}")
    ax_ind.set_xlabel("Predicción")
    ax_ind.set_ylabel("Etiqueta real")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig_ind.savefig(ruta_cm, dpi=150)
    plt.close(fig_ind)

    resultados[nombre] = {
        "accuracy": round(accuracy, 4),
        "precision_macro": round(precision_macro, 4),
        "recall_macro": round(recall_macro, 4),
        "f1_macro": round(f1_macro, 4),
        "confusion_matrix_path": str(ruta_cm.relative_to(BASE_DIR)),
        "classification_report": classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        ),
    }
    modelos_entrenados[nombre] = clf

plt.tight_layout()
plt.show()

## 6. Tabla Comparativa Final y Selección del Mejor Modelo

Ordenamos los modelos por `f1_macro` de mayor a menor para seleccionar el clasificador ganador.

In [ ]:
tabla_comparativa = pd.DataFrame(
    [
        {
            "modelo": nombre,
            "accuracy": datos["accuracy"],
            "precision_macro": datos["precision_macro"],
            "recall_macro": datos["recall_macro"],
            "f1_macro": datos["f1_macro"],
        }
        for nombre, datos in resultados.items()
    ]
).sort_values(by="f1_macro", ascending=False)

modelo_ganador = str(tabla_comparativa.iloc[0]["modelo"])
mejor_f1 = float(tabla_comparativa.iloc[0]["f1_macro"])

print(f"Modelo ganador seleccionado: {modelo_ganador} (F1-score macro = {mejor_f1:.4f})")
tabla_comparativa

## 7. Exportación de Artefactos (`models/vectorizador.pkl` y `models/modelo.pkl`)

Exportamos por separado el vectorizador TF-IDF y el clasificador ganador usando `joblib.dump` para que la aplicación de Streamlit (`app.py`) pueda cargarlos directamente.

In [ ]:
joblib.dump(vectorizador, VECTORIZER_OUTPUT_PATH)
joblib.dump(modelos_entrenados[modelo_ganador], MODEL_OUTPUT_PATH)

resumen_metricas = {
    "winner": modelo_ganador,
    "winner_f1_macro": mejor_f1,
    "dataset_size": int(len(df)),
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "classes": etiquetas,
    "models": resultados,
}
with METRICS_OUTPUT_PATH.open("w", encoding="utf-8") as archivo:
    json.dump(resumen_metricas, archivo, ensure_ascii=False, indent=2)

print(f"Vectorizador guardado en: {VECTORIZER_OUTPUT_PATH}")
print(f"Modelo ganador guardado en: {MODEL_OUTPUT_PATH}")
print(f"Reporte de métricas guardado en: {METRICS_OUTPUT_PATH}")